In [ ]:
%pip install langchain_ollama
%pip install langchain_community
%pip install faiss-cpu
%pip install langchain_core
%pip install pymupdf

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
local_path = "Introduction to Biology.pdf"
loader = PyMuPDFLoader(file_path=local_path)
data = loader.load()

In [3]:
#Split Document

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough

split_document = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = split_document.split_documents(data)
print(f"Split Document into {len(chunks)} chunks")

Split Document into 113 chunks


In [ ]:
!ollama list

In [8]:
#Call embeddings model

embeddings = OllamaEmbeddings(
    model="nomic-embed-text-v2-moe"
)

In [9]:
vector_db = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings)

print("Vector database created successfully")

Vector database created successfully


In [10]:
local_model = "llama3.2"
llm = ChatOllama(model=local_model)

In [11]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 4}
)

# Prompt
template = """
You are PDFTeacher, an AI learning assistant.

Your task is to answer the user's question ONLY using the provided context.

Rules:
- Use only the information from the context.
- If the answer is not found in the context, say:
  "I couldn't find the answer in the provided document."
- Do not make up information.
- Explain the answer clearly and simply, as if teaching a student.
- Keep the answer concise (3-6 sentences).

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

In [12]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [13]:
from IPython.display import display, Markdown

def chat_with_pdf(question):
  display(Markdown(f"**Answer:**"))
  return display(Markdown(chain.invoke(question)))

In [14]:
chat_with_pdf("apa itu biologi")

**Answer:**

I'll explain the answer clearly and simply.

In this context, "biologi" means Biology in Indonesian. Biologi is the study of living things, such as plants, animals, and microorganisms. It includes various branches like evolutionary biology, cellular biology, genetics, and more. Biologists also try to understand what makes life unique, how organisms change over time, and how they are classified.

In essence, biologi is a science that explores the wonders of living things and helps us understand our world around us.

In [15]:
chat_with_pdf("what is biology?")

**Answer:**

Biology is the study of living things. It's the science that looks at everything related to life, such as how living things are organized, their functions, and patterns. Biology also studies growth, development, and reproduction in living things. Think of it like trying to understand all the different parts that make up a living thing, like a big puzzle!

In [16]:
chat_with_pdf("siapa presiden ke-3 indonesia")

**Answer:**

I couldn't find the answer in the provided document. The document appears to be an introduction to biology and does not contain information about Indonesian presidents.